# tarja, em 5 minutos

EN: find Brazilian personal identifiers in free Portuguese text, with check-digit validation.
PT: acha identificadores pessoais brasileiros em texto livre em português, conferindo o dígito verificador.

Rode as células na ordem. Nada sai deste caderno, e nenhum dado é enviado a lugar nenhum.

Documentação: <https://macmaia.github.io/tarja/> · Código: <https://github.com/macmaia/tarja>


In [ ]:
!pip install -q tarja
import tarja

tarja.__version__

## 1. O problema

Um CPF tem onze dígitos. Em texto administrativo brasileiro, onze dígitos também é protocolo, matrícula e
código interno. Procurar só pelo formato transforma o relatório em ruído.

Os dois últimos dígitos de um CPF são calculados a partir dos nove primeiros, por módulo 11. É isso que
separa um CPF de onze dígitos quaisquer.


In [ ]:
bom = "529.982.247-25"  # DV correto
ruim = "529.982.247-24"  # um digito trocado

print(tarja.validate("BR_CPF", bom))
print(tarja.validate("BR_CPF", ruim))

## 2. Achar

`find()` devolve um objeto por achado, com a posição, o tipo e um score.


In [ ]:
texto = (
    "Paciente Maria Silva, CPF 529.982.247-25, cartao SUS 729 1417 7763 1701. "
    "Processo 0000001-83.2017.8.26.0100. Protocolo interno 12345678901."
)

for m in tarja.find(texto):
    print(f"{m.entity:12} {m.score:.2f}  {texto[m.start : m.end]}")

Repare que o protocolo interno de onze dígitos **não** aparece. Ele tem o formato de um CPF, mas não passa
no dígito verificador, então não é reportado. Esse é o ponto central da biblioteca.


## 3. Mascarar

Três estratégias, com propriedades diferentes.


In [ ]:
print(tarja.mask(texto))

In [ ]:
# mesmo valor -> mesmo rotulo, dentro deste texto
print(tarja.mask(texto, strategy="pseudonym"))

In [ ]:
# mesmo valor -> mesmo rotulo em TODO documento, sob a mesma chave
# a chave e segredo: em producao vem de um gestor de segredo, nunca do codigo
print(tarja.mask(texto, strategy="pseudonym_stable", salt="chave-de-exemplo-nao-use-em-producao"))

Os 4 caracteres no meio do rótulo são o **marcador de geração da chave**, derivados da própria chave.
Mesma chave, mesmo marcador, em qualquer máquina. Chave diferente, marcador diferente. Serve para você
saber se dois documentos mascarados podem ou não ser comparados entre si.


## 4. Mandar texto para um LLM sem mandar o dado pessoal

O `Vault` troca cada identificador por um token reversível que só existe na memória do seu processo.


In [ ]:
vault = tarja.Vault()
seguro = vault.protect(texto)
print(seguro)
print()
print("sobrou alguma coisa?", tarja.residual(seguro))

In [ ]:
import re

# o LLM responde usando os tokens, e voce reidrata a resposta
token = re.search(r"<BR_CPF:[^>]+>", seguro).group()
resposta_do_llm = f"O paciente de CPF {token} deve retornar em 30 dias."
print(resposta_do_llm)
print(vault.reveal(resposta_do_llm, issued_by=seguro))

O `reveal()` só resolve tokens emitidos por aquela chamada de `protect()`. Um token ecoado do texto de
outra pessoa não resolve, mesmo que o mesmo cofre atenda vários usuários. O escopo é de uso único e vence
em uma hora.


## 5. Dígito errado é sinal, não lixo

Número com cara de identificador e dígito verificador errado costuma ser erro de digitação na origem ou
ruído de OCR. O `report_invalid=True` traz esses casos marcados.

Atenção: um suspeito é quase-dado-pessoal, não é saída de depuração. Não jogue em log.


In [ ]:
for m in tarja.find("cpf 529.982.247-24", report_invalid=True):
    print(m.entity, m.valid_dv, m.score)

## 6. A sua própria entidade

Identificador interno da sua empresa, sem precisar de fork.


In [ ]:
tarja.register_entity(
    "ACME_EMPLOYEE_ID",
    [("acme", r"\bAC-\d{6}\b", 0.3)],
    context_words=["matricula acme"],  # minusculo, sem acento
)

print(tarja.find("matricula acme AC-123456")[0].entity)

## 7. Pela linha de comando

A instalação traz o comando `tarja`, para arquivo ou entrada padrão.


In [ ]:
!echo 'CPF 529.982.247-25 e protocolo 12345678901' > /tmp/exemplo.txt
!tarja scan /tmp/exemplo.txt --format table
!tarja mask /tmp/exemplo.txt

## O que ele não faz

Não detecta nome, endereço nem dado clínico em prosa: isso depende de reconhecimento de entidade nomeada e
não está implementado. Ausência de achado não é prova de ausência de dado pessoal. E dígito verificador
válido não quer dizer que o número pertence a alguém real, só que é bem formado.

Se você já usa o Presidio, veja [Presidio em português](https://macmaia.github.io/tarja/presidio-em-portugues.html).

Apache 2.0. Contato: tarja@micah6ai.com
